# Customer Segmentation: Unsupervised Learning with KMeans

**Dataset:** Mall Customer Segmentation Dataset (200 customers)  
**Scope:** Exploratory Analysis · Preprocessing · PCA · Elbow Method · KMeans Clustering · Business Interpretation  
**Algorithm:** KMeans (k=5)

---

This notebook builds a complete customer segmentation pipeline, from raw data exploration to actionable business profiles. Unlike supervised learning, no labels are used: KMeans discovers the natural structure of the data on its own.

## Table of Contents

1. [Library Imports](#section-1)
2. [Data Loading](#section-2)
3. [Exploratory Data Analysis](#section-3)
4. [Visualizations](#section-4)
5. [Preprocessing](#section-5)
6. [PCA: Dimensionality Reduction](#section-6)
7. [Elbow Method: Choosing the Number of Clusters](#section-7)
8. [KMeans Clustering](#section-8)
9. [Business Analysis of Segments](#section-9)

---
## 1 · Library Imports

The stack is lean by design: this project is purely classical machine learning with no deep learning. Four Scikit-learn components are imported: `StandardScaler` for normalization, `PCA` for dimensionality reduction, `KMeans` for clustering, and `silhouette_score` to evaluate cluster quality.

In [ ]:
import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

---
## 2 · Data Loading

The dataset contains 200 customers with 5 columns: `CustomerID` (sequential ID with no analytical value), `Gender`, `Age`, `Annual Income (k$)`, and `Spending Score (1-100)`. The spending score is the most distinctive variable: it reflects actual purchasing behavior rather than earning capacity, and is assigned internally by the mall based on visit frequency and spend per visit.

In [ ]:
df = pd.read_csv("Mall_Customers.csv")

print(df.shape)
df.head(10)

---
## 3 · Exploratory Data Analysis

A clean dataset: no missing values across any column. Key statistics show an average age of 38 years, average annual income around $60,000, and a spending score centered at 50, indicating a well-balanced distribution between frugal and heavy spenders. The dataset has 112 female and 88 male customers.

In [ ]:
df.info()

df.describe()

df.isnull().sum()

# Distribution by gender
df["Gender"].value_counts()

---
## 4 · Visualizations

Two charts are produced. The age histogram reveals two distinct peaks: a young cluster (20-35) and a middle-aged cluster (40-50), each with different purchasing behaviors. The scatter plot of annual income vs. spending score is the heart of the project: even before any algorithm runs, five natural groupings are already visible to the naked eye. KMeans will formalize what the eye already perceives intuitively.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["Age"], bins=20, color="#1a1916")
plt.title("Age distribution of customers")
plt.xlabel("Age")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Gender",
    palette={"Male": "#1a1916", "Female": "#c8b98a"},
    s=70,
    alpha=0.8
)

plt.title("Annual income vs Spending score")
plt.xlabel("Annual income (k$)")
plt.ylabel("Spending score (1-100)")
plt.tight_layout()
plt.show()

---
## 5 · Preprocessing

Three sequential steps prepare the data for KMeans. Gender is binary-encoded (Male=0, Female=1) since the algorithm requires numbers. `CustomerID` is removed as it carries no behavioral information. Standardization is the most critical step: KMeans relies on Euclidean distances, and without it, annual income (range 15-137) would completely dominate over spending score and age. After standardization, every variable has mean 0 and standard deviation 1, contributing equally to the clustering.

In [ ]:
# Encode gender: Male = 0, Female = 1
df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})

# Remove customer ID (no analytical value)
X = df.drop("CustomerID", axis=1)

print("Retained features:", X.columns.tolist())
print("Shape:", X.shape)

# Standardize all variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Mean after standardization:", X_scaled.mean(axis=0).round(2))
print("Std after standardization:", X_scaled.std(axis=0).round(2))

---
## 6 · PCA: Dimensionality Reduction

With five variables, direct visualization is impossible. PCA projects the data onto the two directions of maximum variance. The `explained_variance_ratio_` shows how much information is retained: on this dataset, the first two components typically capture more than 80% of total variance, making the 2D projection highly representative. This PCA output is reused in section 8 to display the final colored cluster chart.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Variance explained per component:")
print(pca.explained_variance_ratio_)
print(f"Total captured: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
plt.figure(figsize=(10, 7))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    alpha=0.6,
    s=50,
    color="#1a1916"
)

plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("PCA projection of customers (before clustering)")
plt.tight_layout()
plt.show()

---
## 7 · Elbow Method: Choosing the Number of Clusters

KMeans requires k to be specified in advance. The elbow method finds the optimal k by training KMeans for values 1 to 10 and plotting inertia (sum of squared distances from each point to its cluster center) against k. The goal is not to minimize inertia at any cost, but to identify the point where the rate of decrease slows sharply, forming an elbow. On this dataset, the elbow appears clearly at **k=5**, consistent with the five natural groups visible in the scatter plot.

In [ ]:
inertia = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker='o', color="#1a1916", linewidth=2)
plt.xlabel("Number of clusters k")
plt.ylabel("Inertia")
plt.title("Elbow method: optimal number of clusters")
plt.axvline(x=5, color="#c8b98a", linestyle="--", linewidth=1.5, label="k = 5 (elbow)")
plt.legend()
plt.tight_layout()
plt.show()

---
## 8 · KMeans Clustering

KMeans is run with k=5. The `n_init=10` parameter re-runs the algorithm 10 times with different random initializations and keeps the best result, reducing the risk of converging to a local minimum. Each customer is assigned a cluster label (0 to 4), which is stored back in the dataframe. The final scatter plot combines the PCA projection with cluster colors: well-separated color regions confirm that the clustering is meaningful. The silhouette score quantifies this separation on a scale from -1 to 1; a score around 0.55 is expected on this dataset.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df["Cluster"] = clusters

print("Number of customers per cluster:")
print(df["Cluster"].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(10, 7))

sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=df["Cluster"],
    palette="tab10",
    s=80,
    alpha=0.85
)

plt.title("Customer segments: KMeans (k=5) projected onto PCA")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
score = silhouette_score(X_scaled, clusters)
print(f"Silhouette score: {score:.4f}")

---
## 9 · Business Analysis of Segments

Cluster numbers carry no meaning for a business decision maker. This final step translates the numerical output into named customer profiles by computing the average of each variable per cluster. Five recurring profiles emerge from this dataset:

- **Premium customers:** high income, high spending score. Priority targets for VIP offers and early access to new collections.
- **Cautious customers:** high income, low spending score. Strong untapped potential; respond better to quality and value arguments than to promotions.
- **Standard customers:** average income, average spending score. The core of the customer base; well served by loyalty programs and seasonal deals.
- **Impulsive customers:** low income, high spending score. Very reactive to flash sales, limited-time promotions, and in-store impulse triggers.
- **Frugal customers:** low income, low spending score. Visit the mall but rarely buy; aggressive entry-level offers may convert them into regular buyers.

In [ ]:
profil = df.groupby("Cluster")[
    ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
].mean().round(1)

print(profil)

---
## Conclusion

This notebook demonstrated a complete unsupervised learning pipeline applied to customer segmentation:

- 200-customer dataset explored and visualized with no missing values
- Gender binary-encoded, CustomerID removed, all features standardized
- PCA reduced five dimensions to two while retaining over 80% of variance
- Elbow method identified k=5 as the optimal number of clusters
- KMeans trained and clusters visualized in PCA space with silhouette evaluation
- Cluster numbers translated into five named business profiles with concrete marketing recommendations

This type of work is used daily in marketing teams, e-commerce platforms, and recommendation systems. Possible next steps: DBSCAN (does not require k to be specified in advance), hierarchical clustering for comparison, or adding RFM features (Recency, Frequency, Monetary) to enrich the customer profiles.